# 🧠 HRM-Text: Hierarchical Reasoning Model for Language Modelling

> **Based on:** *"HRM-Text: Efficient Pretraining Beyond Scaling"* (2025) — [arxiv.org/abs/2605.20613](https://arxiv.org/abs/2605.20613)

---

## What is HRM-Text?

A standard Transformer processes every token in **one single forward pass** — the model sees the input once and produces an output. HRM-Text proposes something different: instead of going deeper (more layers) or wider (more parameters), it makes the model **think in loops**.

The key idea is to run two lightweight modules in a **nested recurrent loop**, inspired by how the human brain alternates between fast, local processing and slow, high-level planning:

```
  ┌─────────────────────────────────────────────────────────┐
  │              One Forward Pass of HRM-Text               │
  │                                                         │
  │  Input Tokens                                           │
  │       │                                                 │
  │   [Embedding]                                           │
  │       │                                                 │
  │  ┌────▼──────────────────────────────────┐              │
  │  │        Dual-Timescale Loop            │              │
  │  │                                       │              │
  │  │  for each H_cycle (outer loop):       │              │
  │  │    for each L_cycle (inner loop):     │              │
  │  │      z_L = L_net(z_L + z_H)  ← FAST  │              │
  │  │    z_H = H_net(z_H + z_L)    ← SLOW  │              │
  │  └───────────────────────────────────────┘              │
  │       │                                                 │
  │   [LM Head]                                             │
  │       │                                                 │
  │  Output Logits                                          │
  └─────────────────────────────────────────────────────────┘
```

With `H_cycles=2, L_cycles=3`, the execution order is: **L L L → H → L L L → H** (8 total module calls, same parameter count as a standard Transformer).

---

## Why is this better?

| Property | Standard Transformer | HRM-Text |
|---|---|---|
| Reasoning depth | Fixed (one pass) | Dynamic (iterative refinement) |
| Parameter count | N layers | N layers (same!) |
| Compute | O(N) | O(H_cycles × L_cycles × N) |
| Instruction handling | Causal masking | Bidirectional PrefixLM |

The model achieves richer reasoning **without adding parameters** — it reuses the same H and L modules in each cycle, similar to a weight-tied RNN.

## ⚙️ Step 1: Environment Setup

First, let's verify we have a GPU available and clone the project repository.

In [ ]:
# Verify GPU availability
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU detected: {gpu_name}")
    print(f"   Total VRAM : {gpu_mem:.1f} GB")
else:
    print("⚠️  No GPU found. Go to Runtime → Change runtime type → T4 GPU.")

print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Clone the repository
!git clone https://github.com/wisnunugroho21/nugie-hrm-text.git
%cd nugie-hrm-text
!ls

In [ ]:
# Install required packages
# - datasets  : loads the official HRM-Text pretraining data from HuggingFace
# - transformers : provides the official HRM-Text-1B tokenizer
!pip install -q datasets transformers

## 🏗️ Step 2: Architecture Deep Dive

Let's walk through each module of the codebase before running anything.

### Module Map

```
HRMText  (hrm_text.py)              ← Top-level model you'll interact with
└── HierarchicalReasoningModel  (hrm.py)   ← The dual-timescale recurrent loop
    ├── L_net: ReasoningModule  (reasoning_module.py)   ← Fast, fine-grained
    │   └── [N × TransformerBlock  (transformer.py)]
    │           ├── SigmoidGatedAttention  (attention.py)
    │           │   └── RotaryPositionalEmbedding  (rope.py)
    │           └── SwiGLUFeedForward  (swiglu.py)
    └── H_net: ReasoningModule  (same structure)        ← Slow, high-level
```

### Key Design Choices

**1. MagicNorm** — Each `ReasoningModule` ends with a boundary `RMSNorm`. This single norm serves a dual purpose:
- *Forward stability*: re-normalises hidden states between recurrent steps so variance doesn't explode.
- *Backward stability*: gradients flow through PreNorm residual shortcuts, avoiding vanishing gradients.

**2. Grouped Query Attention (GQA)** — `num_kv_heads < num_heads`. Multiple Q-heads share one K/V-head pair. This cuts the KV cache memory by `num_heads / num_kv_heads` with minimal accuracy loss.

**3. Sigmoid Gate** — The attention output is multiplied element-wise by `sigmoid(gate)`, a learned per-token filter. This lets the model dynamically suppress irrelevant context, like an LSTM gate.

**4. RoPE** — Rotary Positional Embeddings encode relative distance directly into Q·K dot products. No learned position table needed.

**5. SwiGLU FFN** — The feed-forward uses `SiLU(gate) × value` instead of plain GELU/ReLU, giving more expressive non-linearity per parameter.

**6. Truncated BPTT with Warmup** — Training through the full recurrent loop is expensive. Only the last `K` steps are backpropagated. `K` starts at 2 and linearly grows to 5 over the first 20% of training — a curriculum that stabilises noisy early-stage gradients.

## 🔬 Step 3: Smoke Test (Forward + Backward Pass)

Before doing any real training, let's verify the model builds correctly, produces the right output shapes, and that gradients flow through the TBPTT window.

In [ ]:
import torch
import torch.nn.functional as F
from hrm_text import HRMText

torch.manual_seed(42)
IGNORE_LABEL = -100  # PyTorch's default cross-entropy ignore index

# ── Tiny model — runs in seconds on CPU or GPU ──────────────────────────────
model = HRMText(
    vocab_size    = 256,
    hidden_size   = 64,
    seq_len       = 32,
    num_heads     = 4,
    num_kv_heads  = 2,
    H_layers      = 2,
    L_layers      = 2,
    H_cycles      = 2,
    L_cycles      = 3,
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

# ── Dummy instruction–response batch ───────────────────────────────────────
# B=2 examples, T=16 tokens each, first 6 tokens = instruction
B, T, prefix_len = 2, 16, 6
input_ids   = torch.randint(0, 256, (B, T))
labels      = input_ids.clone()
labels[:, :prefix_len] = IGNORE_LABEL  # instruction tokens are NOT supervised
prefix_lens = torch.full((B,), prefix_len, dtype=torch.long)

# ── TBPTT schedule ─────────────────────────────────────────────────────────
# bp_steps controls how many recurrent steps receive gradients.
# At step 0 it equals bp_min_steps (2); it grows as training progresses.
total_training_steps = 100_000
current_step         = 0
bp_steps = model.hrm.compute_bp_steps(current_step, total_training_steps)
print(f"TBPTT window at step {current_step}/{total_training_steps}: {bp_steps} steps")

# ── Forward pass ───────────────────────────────────────────────────────────
logits = model(input_ids, prefix_lens=prefix_lens, bp_steps=bp_steps)
print(f"Logits shape: {logits.shape}  (expected: [B=2, T=16, vocab=256])")

# ── Loss (only over response tokens) ───────────────────────────────────────
loss = F.cross_entropy(
    logits.view(-1, 256).float(),
    labels.view(-1).long(),
    ignore_index=IGNORE_LABEL,
)
print(f"Loss: {loss.item():.4f}  (random init ≈ log(256) = {torch.log(torch.tensor(256.0)):.2f})")

# ── Backward pass ──────────────────────────────────────────────────────────
loss.backward()
print("Backward pass: ✅ OK")

## 📈 Step 4: Visualise the TBPTT Warmup Schedule

The TBPTT window `K` starts small so early-stage gradients are stable, then grows as the recurrent states mature. Here's how it ramps up over training.

In [ ]:
import matplotlib.pyplot as plt
from hrm_text import HRMText

# Create a model just to use its compute_bp_steps helper
_m = HRMText(vocab_size=256, hidden_size=64, seq_len=32,
             bp_min_steps=2, bp_max_steps=5, bp_warmup_ratio=0.2)

total_steps = 100_000
steps       = list(range(0, total_steps, 500))
bp_values   = [_m.hrm.compute_bp_steps(s, total_steps) for s in steps]

plt.figure(figsize=(10, 4))
plt.plot(steps, bp_values, color='steelblue', linewidth=2)
plt.axvline(total_steps * 0.2, color='orange', linestyle='--', label='Warmup end (20%)')
plt.xlabel('Training Step')
plt.ylabel('bp_steps (TBPTT window K)')
plt.title('Truncated BPTT Warmup Schedule')
plt.yticks([2, 3, 4, 5])
plt.legend()
plt.tight_layout()
plt.show()

print("K starts at 2 (small, stable) → grows to 5 (full gradient depth) over first 20% of training.")

## 🖥️ Step 5: Model Configuration for T4 (16 GB VRAM)

The T4 has 16 GB of VRAM — enough for a small-to-medium HRM-Text model. Below we define a **"Small" config** suitable for experimenting and learning on a single T4:

| Config | `hidden_size` | Layers (H+L) | Params | VRAM (est.) |
|--------|-------------|-------------|--------|-------------|
| **Small (this notebook)** | 512 | 4+4 | ~35M | ~4 GB |
| Medium | 1024 | 8+8 | ~200M | ~10 GB |
| Paper 1B | 1536 | 16+16 | ~1B | multi-GPU |

The Small config comfortably fits in the T4 and trains at a reasonable speed.

In [ ]:
import torch
from hrm_text import HRMText

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {DEVICE}")

# ── Small model — fits in 16 GB T4 VRAM ────────────────────────────────────
# vocab_size = 65536 matches the official HRM-Text-1B tokenizer.
# Reduce to 32000 if using a different tokenizer (e.g. LLaMA's).
model = HRMText(
    vocab_size   = 65_536,  # official HRM-Text tokenizer vocabulary
    hidden_size  = 512,
    seq_len      = 2048,
    num_heads    = 8,
    num_kv_heads = 4,       # GQA: 2 Q-heads share each KV-head
    H_layers     = 4,       # transformer layers in the slow H module
    L_layers     = 4,       # transformer layers in the fast L module
    H_cycles     = 2,       # outer H cycles per forward pass
    L_cycles     = 3,       # inner L cycles per H cycle  → 8 total module calls
    bp_min_steps = 2,
    bp_max_steps = 5,
    bp_warmup_ratio = 0.2,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}  ({total_params/1e6:.1f}M)")

if DEVICE.type == 'cuda':
    # Measure how much VRAM the model itself occupies
    model_mem = sum(p.numel() * p.element_size() for p in model.parameters()) / 1e9
    print(f"Model weights: ~{model_mem:.2f} GB")
    print(f"VRAM free    : {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

## 📦 Step 6: Load the Official HRM-Text Dataset

The paper's pretraining data is hosted on HuggingFace as [`sapientinc/HRM-Text-data-io-cleaned-20260515`](https://huggingface.co/datasets/sapientinc/HRM-Text-data-io-cleaned-20260515). Each row has:

- `instruction` — the question or problem statement
- `response` — the expected answer
- `condition` — a tag like `"direct"`, `"cot"` (chain-of-thought), `"synth,cot"`, etc.

The `HRMTextDataset` class in `pretraining.py` tokenizes each row using the official `sapientinc/HRM-Text-1B` tokenizer and formats it as:

```
<|im_start|> [condition] [instruction] <|im_end|>   [response] <eos>
└────────────────── prefix (instruction) ──────────┘└── response ──┘
```

We use `max_samples=2000` here to keep the demo fast. Remove the limit for a full pretraining run.

In [ ]:
from transformers import AutoTokenizer

print("Loading tokenizer from sapientinc/HRM-Text-1B …")
tokenizer = AutoTokenizer.from_pretrained("sapientinc/HRM-Text-1B")

print(f"Vocabulary size : {tokenizer.vocab_size:,}")
print(f"EOS token       : '{tokenizer.eos_token}' (id={tokenizer.eos_token_id})")

# Use pad token if available, otherwise fall back to EOS.
pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
print(f"Pad token id    : {pad_id}")

In [ ]:
from pretraining import HRMTextDataset

# Load 2000 examples for a quick demo run.
# Set max_samples=None to use the full dataset for real pretraining.
dataset = HRMTextDataset(
    tokenizer   = tokenizer,
    split       = "train",
    max_seq_len = 2048,
    max_samples = 2000,
)

print(f"\nDataset size: {len(dataset):,} examples")

# Inspect the first example
example = dataset[0]
ids  = example["input_ids"]
plen = example["prefix_len"]
print(f"\nExample 0:")
print(f"  Sequence length : {ids.size(0)} tokens")
print(f"  Prefix (instr.) : {plen} tokens")
print(f"  Response tokens : {ids.size(0) - plen} tokens")
print(f"\nInstruction (decoded):")
print(tokenizer.decode(ids[:plen].tolist()))
print(f"\nResponse (first 100 tokens):")
print(tokenizer.decode(ids[plen:plen+100].tolist()))

## 🚀 Step 7: Pretraining

Now we put it all together. Here's what happens in each training step:

1. **Compute `bp_steps`** — the TBPTT window for this step via the warmup schedule.
2. **Forward pass** — `HRMText` builds the `PrefixLM` attention mask from `prefix_lens` and runs the recurrent H/L loop.
3. **Task-completion loss** — Cross-entropy over response tokens only; instruction and padding tokens are masked out with `IGNORE_LABEL = -100`.
4. **Backward + gradient clip + optimizer step** — Standard AdamW with gradient clipping.
5. **EMA update** — Shadow weights updated with `decay = 0.9999`.
6. **LR schedule** — Linear warmup → cosine decay.

> **T4 tip:** `batch_size=4` and `seq_len=2048` is safe for 16 GB. Increase `batch_size` cautiously — if you hit OOM, reduce it or `max_seq_len`.

In [ ]:
from pretraining import PretrainConfig, train

cfg = PretrainConfig(
    # ── Architecture (must match the model built in Step 5) ─────────────────
    vocab_size        = tokenizer.vocab_size,  # 65 536
    hidden_size       = 512,
    num_heads         = 8,
    num_kv_heads      = 4,
    H_layers          = 4,
    L_layers          = 4,
    H_cycles          = 2,
    L_cycles          = 3,
    max_seq_len       = 2048,
    pad_token_id      = pad_id,

    # ── Optimizer ────────────────────────────────────────────────────────────
    # global_batch_size=4 is safe for the T4's 16 GB VRAM with seq_len=2048.
    # For faster training with gradient accumulation, see the note below.
    global_batch_size = 4,
    epochs            = 1,
    lr                = 3e-4,
    lr_warmup_steps   = 100,  # short warmup for this demo
    beta1             = 0.9,
    beta2             = 0.95,
    weight_decay      = 0.1,
    grad_clip         = 1.0,
    ema_decay         = 0.9999,

    # ── TBPTT schedule ───────────────────────────────────────────────────────
    bp_warmup_ratio   = 0.2,
    bp_min_steps      = 2,
    bp_max_steps      = 5,

    # ── Logging & Checkpoints ────────────────────────────────────────────────
    log_interval         = 10,
    checkpoint_interval  = 1,
    checkpoint_dir       = "checkpoints",
)

steps_per_epoch = len(dataset) // cfg.global_batch_size
print(f"Steps per epoch : {steps_per_epoch}")
print(f"Total steps     : {steps_per_epoch * cfg.epochs}")
print(f"\nConfig ready. Run the cell below to start training.")

In [ ]:
# ── Start pretraining ───────────────────────────────────────────────────────
# train() returns the trained model and its EMA shadow copy.
# Progress is printed every log_interval steps.

model, ema = train(cfg, dataset)
print("\n✅ Pretraining complete.")

## 💾 Step 8: Save & Resume from Checkpoints

Checkpoints are saved automatically at the end of each epoch (or every `checkpoint_interval` epochs). They include the model weights, optimizer state, EMA shadow, and config.

To resume a run that was interrupted, pass `resume_from` to `train()`.

In [ ]:
import os

ckpt_dir = cfg.checkpoint_dir
checkpoints = sorted([
    f for f in os.listdir(ckpt_dir) if f.endswith(".pt")
]) if os.path.isdir(ckpt_dir) else []

if checkpoints:
    latest = os.path.join(ckpt_dir, checkpoints[-1])
    ckpt   = torch.load(latest, map_location="cpu")
    print(f"Latest checkpoint : {latest}")
    print(f"  Epoch : {ckpt['epoch']}")
    print(f"  Step  : {ckpt['step']}")
    print(f"\nTo resume training from this checkpoint:")
    print(f"  model, ema = train(cfg, dataset, resume_from='{latest}')")
else:
    print("No checkpoints found yet — run Step 7 first.")

## 🤖 Step 9: Greedy Inference

After training, we can run the model in inference mode. Below is a simple **greedy decoding** loop — at each step we pick the token with the highest logit.

> For better generation quality, replace greedy with **temperature sampling** or **top-k/top-p** sampling.

In [ ]:
import torch

def greedy_generate(
    model,
    tokenizer,
    instruction: str,
    max_new_tokens: int = 100,
    device = DEVICE,
):
    """
    Greedy autoregressive generation for one instruction.

    The instruction is wrapped with <|im_start|> ... <|im_end|> to match
    the format seen during pretraining. The model then generates response
    tokens one by one using causal attention.
    """
    model.eval()

    # Build the instruction prefix exactly as the dataset does
    prefix_text = f"<|im_start|>{instruction}<|im_end|>"
    prefix_ids  = tokenizer.encode(prefix_text, add_special_tokens=False)
    prefix_len  = len(prefix_ids)

    ids = torch.tensor([prefix_ids], dtype=torch.long, device=device)  # [1, prefix_len]

    with torch.no_grad():
        for _ in range(max_new_tokens):
            prefix_lens_t = torch.tensor([prefix_len], dtype=torch.long, device=device)

            # During inference bp_steps doesn't matter for correctness;
            # we pass a default value of 5.
            logits = model(ids, prefix_lens=prefix_lens_t, bp_steps=5)  # [1, T, V]

            # Greedy: pick the highest-probability token at the last position
            next_token = logits[0, -1, :].argmax(dim=-1, keepdim=True)  # [1]
            ids = torch.cat([ids, next_token.unsqueeze(0)], dim=1)      # append

            # Stop at EOS
            if tokenizer.eos_token_id is not None and next_token.item() == tokenizer.eos_token_id:
                break

    response_ids = ids[0, prefix_len:].tolist()
    return tokenizer.decode(response_ids, skip_special_tokens=True)


# ── Example generation ──────────────────────────────────────────────────────
# Note: a model trained for only 1 epoch on 2000 samples will produce
# low-quality text. This cell just demonstrates the generation pipeline.
instruction = "What is 2 + 2?"
print(f"Instruction: {instruction}")
print("Response    :", greedy_generate(model, tokenizer, instruction, max_new_tokens=50))

## 🗂️ Step 10 (Bonus): Training on Your Own Data

You can train on any instruction–response dataset using `InstructionDataset`. Just provide a list of `(instruction_ids, response_ids)` pairs where each element is a **list of integer token IDs**.

In [ ]:
from pretraining import InstructionDataset, PretrainConfig, train

# ── Example: tiny hand-crafted arithmetic dataset ───────────────────────────
# In practice, replace this with your own tokenized instruction–response pairs.

qa_pairs = [
    ("What is 2 + 2?",  "4"),
    ("What is 3 * 5?",  "15"),
    ("What is 10 - 4?", "6"),
    ("What is 8 / 2?",  "4"),
    ("What is 7 + 3?",  "10"),
    ("What is 6 * 6?",  "36"),
]

# Tokenize all pairs
tokenized = [
    (
        tokenizer.encode(q, add_special_tokens=False),
        tokenizer.encode(a, add_special_tokens=False) + [tokenizer.eos_token_id],
    )
    for q, a in qa_pairs
]

custom_dataset = InstructionDataset(tokenized, max_seq_len=128)
print(f"Custom dataset size: {len(custom_dataset)} examples")

# Inspect the first example
ex = custom_dataset[0]
print(f"Sequence: {tokenizer.decode(ex['input_ids'].tolist())}")
print(f"Prefix length: {ex['prefix_len']} tokens (instruction)")

In [ ]:
# Train a tiny model on the custom dataset
tiny_cfg = PretrainConfig(
    vocab_size        = tokenizer.vocab_size,
    hidden_size       = 128,
    num_heads         = 4,
    num_kv_heads      = 2,
    H_layers          = 2,
    L_layers          = 2,
    H_cycles          = 2,
    L_cycles          = 3,
    max_seq_len       = 128,
    pad_token_id      = pad_id,
    global_batch_size = 2,
    epochs            = 5,         # more epochs to overfit this tiny dataset
    lr                = 1e-3,
    lr_warmup_steps   = 10,
    grad_clip         = 1.0,
    ema_decay         = None,      # disable EMA for this tiny demo
    log_interval      = 5,
    checkpoint_dir    = "checkpoints_custom",
)

tiny_model, _ = train(tiny_cfg, custom_dataset)
print("\n✅ Custom data training complete.")

## 📝 Summary

Here's what we covered in this notebook:

| Step | What we did |
|------|-------------|
| 1 | Verified GPU, cloned repo, installed dependencies |
| 2 | Walked through the full architecture (H/L loop, GQA, MagicNorm, RoPE, SwiGLU) |
| 3 | Ran a smoke test — forward + backward pass, verified shapes |
| 4 | Visualised the Truncated BPTT warmup schedule |
| 5 | Defined a Small model config safe for the T4's 16 GB VRAM |
| 6 | Loaded the official HRM-Text dataset and tokenizer from HuggingFace |
| 7 | Ran pretraining with the `train()` function |
| 8 | Demonstrated checkpoint save and resume |
| 9 | Ran greedy inference on the trained model |
| 10 | Showed how to train on a custom instruction–response dataset |

### Next Steps

- **Scale up**: set `max_samples=None` and increase `hidden_size` / `num_layers` to train a larger model.
- **Better inference**: replace greedy decoding with temperature sampling or beam search.
- **Export**: save the EMA weights with `torch.save(ema.state_dict(), 'ema_weights.pt')` for evaluation.
- **Paper**: read the original paper at [arxiv.org/abs/2605.20613](https://arxiv.org/abs/2605.20613) for full details.

---
*HRM-Text — Efficient Pretraining Beyond Scaling (2025)*